In [1]:
import numpy as np
import pandas as pd
import sqlite3
import json
from pathlib import Path

# Task 1

In [2]:
np.random.seed(42)
n = 400

registrations = pd.DataFrame({
    'reg_id': range(1, n + 1),
    'student_id': ['S' + str(i).zfill(3) for i in range(1, n + 1)],
    'event': np.random.choice(['Dance', 'Coding', 'Quiz', 'Gaming', 'Music'], n),
    'college': np.random.choice(['Sunrise College', 'Riverdale Institute', 'Orchid University', 'Lotus College', 'Pinnacle Institute', 'Heritage College'], n),
    'ticket_inr': np.random.choice(range(0, 501, 50), n),
    'paid': np.random.choice(['Yes', 'No'], n)
})

conn = sqlite3.connect('fest.db')
registrations.to_sql('registrations', conn, if_exists='replace', index=False)

query = "SELECT * FROM registrations WHERE paid = 'Yes'"
paid_query = pd.read_sql_query(query, conn)
paid_sql = pd.read_sql(query, conn)

print("read_sql_query shape:", paid_query.shape)
print("read_sql shape:", paid_sql.shape)

read_sql_query shape: (210, 6)
read_sql shape: (210, 6)


# Task 2

In [3]:
sponsor_names = ['TechNova', 'GreenLeaf', 'BlueWave', 'SkyBridge', 'UrbanCart', 'FreshBite', 'CodeCraft', 'MegaMart', 'SunPower', 'QuickPay', 'BrightEdu', 'FitLife', 'TravelEase', 'PrintHub', 'CoolBrew']
cities = ['Ahmedabad', 'Surat', 'Vadodara', 'Rajkot']
tiers = ['Gold', 'Silver', 'Bronze']

data = []
for name in sponsor_names:
    data.append({
        'sponsor': name,
        'amount_inr': int(np.random.randint(10000, 200000)),
        'contact': {
            'city': str(np.random.choice(cities)),
            'tier': str(np.random.choice(tiers))
        }
    })

with open('sponsors_api.json', 'w') as f:
    json.dump({'status': 'ok', 'data': data}, f, indent=2)

with open('sponsors_api.json') as f:
    response = json.load(f)

sponsors = pd.json_normalize(response['data'])
print(sponsors.columns)

Index(['sponsor', 'amount_inr', 'contact.city', 'contact.tier'], dtype='str')


# Task 3

In [4]:
fest_events = pd.DataFrame({
    'event': ['Dance', 'Coding', 'Quiz', 'Gaming', 'Music'],
    'venue': ['Main Auditorium', 'Computer Lab', 'Seminar Hall', 'Gaming Arena', 'Open Air Stage'],
    'day': [1, 2, 2, 3, 3],
    'budget_inr': [50000, 30000, 15000, 40000, 60000]
})
fest_events.to_csv('fest_events.csv', index=False)

github_user = 'your-github-username'
url = 'https://raw.githubusercontent.com/' + github_user + '/ds_analysis_portfolio/main/fest_events.csv'

events = pd.read_csv(url)
print(events.shape)

HTTPError: HTTP Error 404: Not Found

# Task 4

In [5]:
Path('fest_data').mkdir(exist_ok=True)

registrations.iloc[:130].to_csv('fest_data/day1.csv', index=False)
registrations.iloc[130:270].to_csv('fest_data/day2.csv', index=False)
registrations.iloc[270:].to_csv('fest_data/day3.csv', index=False)

files = sorted(Path('fest_data').glob('day*.csv'))
all_days = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

fest_master = all_days.merge(events, on='event', how='left', validate='m:1')
fest_master.to_csv('fest_master.csv', index=False)

assert len(fest_master) == 400
assert fest_master['venue'].notna().all()
print(fest_master.shape)
conn.close()

NameError: name 'events' is not defined